# 🔍 WordFinder
**Keyword-in-context concordancing — search for words and phrases across your texts** — LADAL

The tool starts automatically. If the button does not appear within 90 seconds, run the cell below manually with **Shift + Enter**.

In [ ]:
import subprocess, time, os, socket, tempfile
from IPython.display import display, HTML

port  = 3838
path  = '/home/jovyan/tools/wordfinder'
name  = 'WordFinder'
emoji = '🔍'

# Log file so we can see R errors if Shiny crashes
log_file = os.path.join(tempfile.gettempdir(), 'shiny_wordfinder.log')

display(HTML(f'''
<div style="padding:16px;background:#f4f0f8;
            border-left:4px solid #51247a;border-radius:6px;
            font-family:sans-serif;margin-bottom:12px;">
  <b style="font-size:1.1rem;color:#51247a;">{emoji} {name}</b><br>
  <span style="color:#555;font-size:.9rem;">Keyword-in-context concordancing — search for words and phrases across your texts</span>
</div>
<p style="font-family:sans-serif;color:#555;font-size:.9rem;">
  ⏳ Starting {name}... please wait (up to 90 seconds).
</p>
'''))

# Kill any existing Shiny process on this port to avoid stale sockets
subprocess.run(['fuser', '-k', f'{port}/tcp'],
               capture_output=True, check=False)
time.sleep(1)

# Start Shiny, capturing output to log file for debugging
with open(log_file, 'w') as log:
    proc = subprocess.Popen(
        ['R', '--vanilla', '-e',
         f"shiny::runApp('{path}', port={port}, host='0.0.0.0', launch.browser=FALSE)"],
        stdout=log,
        stderr=subprocess.STDOUT
    )

# Poll until port is open AND stays open (up to 90 seconds)
ready = False
for i in range(90):
    time.sleep(1)
    # Check process is still alive
    if proc.poll() is not None:
        # Process exited — read log and show error
        try:
            with open(log_file) as f:
                log_content = f.read()[-3000:]  # last 3000 chars
        except:
            log_content = '(no log available)'
        display(HTML(f'''
        <div style="padding:16px;background:#fff0f0;
                    border-left:4px solid #e74c3c;border-radius:6px;
                    font-family:sans-serif;">
          <b style="color:#c0392b;">❌ {name} failed to start.</b><br><br>
          <details><summary style="cursor:pointer;color:#c0392b;">Show error log</summary>
          <pre style="background:#fff;padding:10px;border-radius:4px;
                      font-size:.78rem;overflow-x:auto;white-space:pre-wrap;">{log_content}</pre>
          </details>
        </div>
        '''))
        break
    try:
        s = socket.create_connection(('localhost', port), timeout=1)
        s.close()
        ready = True
        break
    except:
        pass

base_url = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
tool_url  = f'{base_url}proxy/{port}/'

if ready:
    display(HTML(f'''
    <div style="padding:16px;background:#eafaf1;
                border-left:4px solid #27ae60;border-radius:6px;
                font-family:sans-serif;">
      <p style="margin:0 0 12px 0;font-size:1rem;">
        ✔ <b>{name} is ready!</b>
      </p>
      <a href="{tool_url}" target="_blank"
         style="display:inline-block;padding:12px 28px;
                background:#51247a;color:white;border-radius:8px;
                text-decoration:none;font-weight:bold;font-size:1rem;">
        {emoji} Open {name}
      </a>
      <p style="margin:10px 0 0 0;font-size:.82rem;color:#888;">
        The tool opens in a new browser tab.
        If you see a 404 page, wait 5 seconds and refresh.
      </p>
    </div>
    '''))
elif proc.poll() is None:
    # Still running but port not open after 90s
    try:
        with open(log_file) as f:
            log_content = f.read()[-2000:]
    except:
        log_content = '(no log)'
    display(HTML(f'''
    <div style="padding:16px;background:#fff4e5;
                border-left:4px solid #f0a500;border-radius:6px;
                font-family:sans-serif;">
      ⚠️ <b>{name} is taking longer than expected.</b><br>
      Re-run this cell (Shift+Enter) to try again.<br><br>
      <details><summary style="cursor:pointer;">Show startup log</summary>
      <pre style="background:#fff;padding:10px;border-radius:4px;
                  font-size:.78rem;overflow-x:auto;white-space:pre-wrap;">{log_content}</pre>
      </details>
    </div>
    '''))
